# Breast Cancer Model Training

This notebook is organized as short, numbered code sections, like the reference screenshot. Each section explains the step before showing its code and output.

The source is the UCI Wisconsin Diagnostic Breast Cancer dataset. The separate CSV contains all **569 samples**, narrowed to the **five measurements used by the Streamlit app**: mean radius, mean texture, mean perimeter, mean area, and mean concavity. `sample_id` is retained for traceability and is not used as a model input.

**Target encoding:** `0 = malignant`, `1 = benign` (212 malignant and 357 benign samples). The notebook evaluates a stratified holdout set, then refits on all rows and saves `model.pkl` and `scaler.pkl` in the project folder. The final save section replaces those artifacts when run.

## Code 1:- Import Libraries and Set Up Paths

Load the data-analysis and machine-learning tools, then locate the prepared CSV. The notebook works from the project folder or with the CSV uploaded beside it.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "training_data.csv"

# Also allow running this notebook after uploading the CSV beside it (for Colab).
if not DATA_PATH.exists() and (PROJECT_DIR / "training_data.csv").exists():
    DATA_PATH = PROJECT_DIR / "training_data.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place training_data.csv in the data folder, or upload it beside this notebook."
    )

FEATURES = [
    "mean_radius",
    "mean_texture",
    "mean_perimeter",
    "mean_area",
    "mean_concavity",
]
TARGET = "target"
print("Dataset:", DATA_PATH)

## Code 2:- Load and Preview the Training Data

Read the labeled samples, check that all model inputs and both target classes are present, and preview the first records.

In [ ]:
data = pd.read_csv(DATA_PATH)
required_columns = FEATURES + [TARGET]
missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if data[required_columns].isna().any().any():
    raise ValueError("The training data contains missing feature or target values.")
if set(data[TARGET].unique()) != {0, 1}:
    raise ValueError("The target column must contain 0 (malignant) and 1 (benign).")

print("Rows and columns:", data.shape)
print("Features:", FEATURES)
display(data.head())

## Code 3:- Basic Statistical Analysis

Calculate the mean, median, population standard deviation, and population variance for every model feature, following the basic statistics format in the reference.

In [ ]:
measurements = data[FEATURES]
summary = pd.DataFrame({
    "Mean": measurements.mean(),
    "Median": measurements.median(),
    "Standard deviation": measurements.std(ddof=0),
    "Variance": measurements.var(ddof=0),
})
display(summary.round(3))
print("\nFull descriptive summary")
display(measurements.describe().round(3))

## Code 4:- Check Class Distribution and Data Quality

Count the benign and malignant samples and check for missing values or repeated sample IDs.

In [ ]:
class_names = {0: "Malignant", 1: "Benign"}
class_counts = data[TARGET].map(class_names).value_counts().rename_axis("Class").to_frame("Samples")
display(class_counts)
print("Missing values in selected columns:", int(data[required_columns].isna().sum().sum()))
print("Duplicate sample IDs:", int(data["sample_id"].duplicated().sum()) if "sample_id" in data else "ID column not included")

## Code 5:- Separate Features and Split the Data

Keep the five app measurements separate from the target label. Use an 80/20 stratified split so both classes are represented in the holdout set.

In [ ]:
X = data[FEATURES].astype(float)
y = data[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## Code 6:- Scale the Feature Values

Fit StandardScaler on the training portion only, then apply that fitted scaler to the holdout portion to avoid data leakage.

In [ ]:
# Fit the scaler only on the training split to avoid data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42, solver="liblinear")
model.fit(X_train_scaled, y_train)
print("Model training complete.")

## Code 7:- Train the Logistic Regression Model

Train the binary classifier on the scaled training samples using a fixed random seed for repeatable results.

In [ ]:
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Malignant", "Benign"],
    digits=3,
))
print("Confusion matrix (rows=true, columns=predicted; order: malignant, benign):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

## Code 8:- Evaluate the Model

Report accuracy, per-class precision and recall, and a confusion matrix for the held-out samples. The matrix order is malignant, then benign.

In [ ]:
# Evaluation above uses an untouched holdout split. After evaluation, refit
# on all labeled rows so the saved deployment model can use the full dataset.
final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X)
final_model = LogisticRegression(max_iter=1000, random_state=42, solver="liblinear")
final_model.fit(X_all_scaled, y)

MODEL_PATH = PROJECT_DIR / "model.pkl"
SCALER_PATH = PROJECT_DIR / "scaler.pkl"
joblib.dump(final_model, MODEL_PATH)
joblib.dump(final_scaler, SCALER_PATH)
print("Saved model:", MODEL_PATH)
print("Saved scaler:", SCALER_PATH)

## Code 9:- Refit on All Data and Save the Artifacts

After holdout evaluation, refit the scaler and classifier on all labeled samples for the app. This section writes the compatible `model.pkl` and `scaler.pkl` files.

In [ ]:
try:
    from google.colab import files
    files.download(str(MODEL_PATH))
    files.download(str(SCALER_PATH))
except ImportError:
    print("Artifacts are saved in the project folder for the Streamlit app.")